In [1]:
SECTION_LIST = ["section1", "section4", "section5", "section6", "section10", "section11", "section12", "section16", 
                "section18", "section19", "section21", "section22", "section23"]

### Section 1: Create a script to convert RDS in markdown format to JSONL dataset

- Create a JSONL file file where each row = one rule    
Schema: standard, edition, section_id, rule_id, title, rule_description, rule_assertion, mandatory_rule, schema_version, evaluation_context, references, data_lookup, rule_logic_steps, source_path, source_sha, variables_defined, data_paths, operations, outcomes, keywords


In [9]:
import re
import json
from pathlib import Path

def extract_rule_info(md_content):    
    def find_rule_id():
        match = re.search(r"Rule[:\s-]*([\w-]+)", md_content, re.IGNORECASE)
        return match.group(1).strip() if match else None

    def find_schema_version():
        match = re.search(r"\*\*Schema Version:\*\*\s*([\w\.-]+)", md_content, re.IGNORECASE)
        if match:
            return match.group(1).strip()
        match = re.search(r"Schema Version:\s*([\w\.-]+)", md_content, re.IGNORECASE)
        return match.group(1).strip() if match else None

    def find_mandatory_rule():
        match = re.search(r"\*\*Mandatory Rule:\*\*\s*([\w]+)", md_content, re.IGNORECASE)
        if match:
            return match.group(1).strip()
        match = re.search(r"Mandatory Rule:\s*([\w]+)", md_content, re.IGNORECASE)
        return match.group(1).strip() if match else None

    def find_rule_description():
        match = re.search(r"\*\*Rule Description:\*\*\s*([\s\S]+?)(?:\n|$)", md_content, re.IGNORECASE)
        if match:
            return match.group(1).strip()
        match = re.search(r"Rule Description:\s*([\s\S]+?)(?:\n|$)", md_content, re.IGNORECASE)
        return match.group(1).strip() if match else None

    def find_rule_assertion():
        match = re.search(r"\*\*Rule Assertion:\*\*\s*([\s\S]+?)(?:\n|$)", md_content, re.IGNORECASE)
        if match:
            return match.group(1).strip()
        match = re.search(r"Rule Assertion:\s*([\s\S]+?)(?:\n|$)", md_content, re.IGNORECASE)
        return match.group(1).strip() if match else None

    def find_appendix_g_section():
        match = re.search(r"\*\*Appendix G Section:\*\*\s*([\s\S]+?)(?:\n|$)", md_content, re.IGNORECASE)
        if match:
            return match.group(1).strip()
        match = re.search(r"Appendix G Section:\s*([\s\S]+?)(?:\n|$)", md_content, re.IGNORECASE)
        if match:
            return match.group(1).strip()
        match = re.search(r"Appendix G Section Reference:\s*([\s\S]+?)(?:\n|$)", md_content, re.IGNORECASE)
        return match.group(1).strip() if match else None

    def find_data_lookup():
        match = re.search(r"\*\*Data Lookup:\*\*\s*([\s\S]+?)(?:\n|$)", md_content, re.IGNORECASE)
        if match:
            return match.group(1).strip()
        match = re.search(r"Data Lookup:\s*([\s\S]+?)(?:\n|$)", md_content, re.IGNORECASE)
        return match.group(1).strip() if match else None

    def find_evaluation_context():
        match = re.search(r"\*\*Evaluation Context:\*\*\s*([\s\S]+?)(?:\n|$)", md_content, re.IGNORECASE)
        if match:
            return match.group(1).strip()
        match = re.search(r"Evaluation Context:\s*([\s\S]+?)(?:\n|$)", md_content, re.IGNORECASE)
        return match.group(1).strip() if match else None

    def find_applicability_checks():
        match = re.search(r"\*\*Applicability Checks:\*\*\s*([\s\S]+?)(?:\n|$)", md_content, re.IGNORECASE)
        if match:
            return match.group(1).strip()
        match = re.search(r"Applicability Checks:\s*([\s\S]+?)(?:\n|$)", md_content, re.IGNORECASE)
        return match.group(1).strip() if match else None

    def find_rule_logic():
        match = re.search(r'(?:\*\*Rule Logic:\*\*|Rule Logic:)\s*([\s\S]+?)(?=\*\*Rule Assertion:\*\*|Rule Assertion:|$)', md_content, re.IGNORECASE)
        return match.group(1).strip() if match else None

    def find_rule_logic_steps():
        # Find the Rule Assertion section after ## Rule Logic, but stop at Notes/Questions, [Back], or next header
        match = re.search(r'## Rule Logic:[\s\S]*?(?:\*\*Rule Assertion:\*\*|Rule Assertion:)\s*([\s\S]+?)(?=\*\*Notes/Questions:\*\*|Notes/Questions:|\\*\\*\[Back\]\(\.\./_toc.md\)\\*\\*|\[Back\]\(\.\./_toc.md\)|^## |\Z)', md_content, re.IGNORECASE | re.MULTILINE)
        return match.group(1).strip() if match else None

    return {
        "rule_id": find_rule_id(),
        "schema_version": find_schema_version(),
        "mandatory_rule": find_mandatory_rule(),
        "rule_description": find_rule_description(),
        "rule_assertion": find_rule_assertion(),
        "Appendix_G_section": find_appendix_g_section(),
        "data_lookup": find_data_lookup(),
        "evaluation_context": find_evaluation_context(),
        "applicability_checks": find_applicability_checks(),
        "rule_logic": find_rule_logic(),
        "rule_logic_steps": find_rule_logic_steps()
    }

def convert_markdown_to_jsonl(md_folders, output_jsonl):
    if isinstance(md_folders, str):
        md_folders = [md_folders]
    with open(output_jsonl, "w", encoding="utf-8") as out_f:
        for md_folder in md_folders:
            md_files = Path(md_folder).glob("*.md")
            for md_file in md_files:
                with open(md_file, "r", encoding="utf-8") as f:
                    content = f.read()
                rule_info = extract_rule_info(content)
                # If fallback to filename, strip 'Rule' prefix if present
                if not rule_info["rule_id"]:
                    stem = md_file.stem
                    rule_id = re.sub(r"^Rule[:\s-]*", "", stem, flags=re.IGNORECASE)
                    rule_info["rule_id"] = rule_id
                out_f.write(json.dumps(rule_info) + "\n")


Convert RDS to JSONL dataset

In [10]:
#md_folders = [f"../../docs/ashrae_90p1_2019/{section}" for section in SECTION_LIST]
md_folders = [f"../../docs/ashrae_90p1_2019/{section}" for section in ["section22"]]
#output_jsonl = "RDS_2019.jsonl"
output_jsonl = "RDS_2019_short.jsonl"
convert_markdown_to_jsonl(md_folders, output_jsonl)
print(f"Conversion complete. Output: {output_jsonl}")

Conversion complete. Output: RDS_2019_short.jsonl


### Section 2: Test out the LLM by feeding the normalized data as instructions/additional info, establish a testing / evaluation framework

##### 2.1 Use the ```sentence-transformers/all-MiniLM-L6-v2``` model

In [1]:
from sentence_transformers import SentenceTransformer, util
import json
import torch

# Load data
jsonl_dataset = "RDS_2019_short.jsonl"

dataset = []
with open(jsonl_dataset, "r", encoding="utf-8") as f:
    for line in f:
        dataset.append(json.loads(line))

# Load model
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# Encode all rule descriptions into embeddings
for r in dataset:
    r["embedding"] = model.encode(r["rule_description"], normalize_embeddings=True)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
dataset

[{'rule_id': '22-1',
  'schema_version': None,
  'mandatory_rule': None,
  'rule_description': 'Baseline chilled water design supply temperature shall be modeled at 44F.',
  'rule_assertion': 'B-RMR = expected value',
  'Appendix_G_section': 'Section 22 CHW&CW Loop',
  'data_lookup': 'None',
  'evaluation_context': 'Building',
  'applicability_checks': '1. B-RMR is modeled with at least one air-side system that is Type-7, 8, 11.1, 11,2, 12, 13, 1a, 3a, 7a, 8a, 11.1a, 11.2a, 12a, 13a, 7b, 8b, 11b, 12b, 1c, 3c, 7c, 11c.',
  'rule_logic': '`if any(sys_type in baseline_hvac_system_dict.keys() for sys_type in ["SYS-7", "SYS-8", "SYS-11.1", "SYS-11.2", "SYS-12", "SYS-13", "SYS-1A", "SYS-3A", "SYS-7A", "SYS-8A", "SYS-11.1A", "SYS-11.2A", "SYS-12A", "SYS-13A", "SYS-7B", "SYS-8B", "SYS-11B", "SYS-12B", "SYS-1C", "SYS-3C", "SYS-7C", "SYS-11C"]): CHECK_RULE_LOGIC`\n\n  - Else, rule is not applicable to B-RMR: `else: RULE_NOT_APPLICABLE`\n\n## Rule Logic:  \n\n- For each boiler in B_RMI, save boil

In [ ]:
# real context in rule logic, context -> need to put in
# "document" 
# extract context and send back to LLMs
# embbding pipline should be separated from retrivel pipeline
# retrivel pipeline - should be used for everytime user asks a question (separate into 2 notebooks) # export to pandas 


41

# 

In [ ]:
# Question
#question = "Which rule sets chilled water supply temperature to 44F?" # expect rule 22-1
question ="Which rule sets baseline heat-rejection device to have a design temperature rise of 10F?" # expect rule 22-14
question = "What are the rules for the baseline heat rejection device?"

# Encode the question
question_embedding = model.encode(question, normalize_embeddings=True)

rule_embeddings = torch.tensor([r["embedding"] for r in dataset])
cos_scores = util.cos_sim(question_embedding, rule_embeddings)[0]

# 8️Find the most similar rule
top_idx = torch.argmax(cos_scores) # ask copilot to pick up top k (e.g., top 5)
top_rule = dataset[top_idx]

print("\n\n")
print("Most relevant rule ID:", top_rule["rule_id"])
print("Rule description:", top_rule["rule_description"])
print("Similarity score:", cos_scores[top_idx].item())




Most relevant rule ID: 22-14
Rule description: The baseline heat-rejection device shall have a design temperature rise of 10°F.
Similarity score: 0.8971464037895203


# LLM to translate user request into standardized question format e.g., extract relevant questions, core questions user asks
# e.g., requirements for heat rejections -> 
# retrival -> doesn't have to be 1:1. use "top_k"

##### 2.2 Use the ```intfloat/e5-base-v2``` model

In [129]:
from sentence_transformers import SentenceTransformer, util
import json
import torch

# Load data
jsonl_dataset = "RDS_2019_short.jsonl"

dataset = []
with open(jsonl_dataset, "r", encoding="utf-8") as f:
    for line in f:
        dataset.append(json.loads(line))

# Load the e5-base-v2 model
model = SentenceTransformer("intfloat/e5-base-v2")

# Encode all rule descriptions
for r in dataset:
    r["embedding"] = model.encode(r["rule_description"], normalize_embeddings=True)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: intfloat/e5-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### 

In [ ]:
# question
#question = "Which rule sets chilled water supply temperature to 44F?" # expect rule 22-1
#question ="Which rule sets baseline heat-rejection device to have a design temperature rise of 10F?" # expect rule 22-14
question = "What are the rules for the baseline heat rejection device?"

# Encode the question
question_embedding = model.encode(question, normalize_embeddings=True)

# Compute cosine similarity
rule_embeddings = torch.tensor([r["embedding"] for r in dataset])
cos_scores = util.cos_sim(question_embedding, rule_embeddings)[0]

# Find the top rule
#top_idx = torch.argmax(cos_scores)
#top_rule = dataset[top_idx]

# Find the top k most similar rules
k = 10  # Set the number of top results you want
top_k_indices = torch.topk(cos_scores, k=min(k, len(dataset))).indices


#print("Most relevant rule ID:", top_rule["rule_id"])
#print("Rule description:", top_rule["rule_description"])
#print("Similarity score:", cos_scores[top_idx].item())

print("\n\n")
print(f"Top {k} most relevant rules:")
print("=" * 80)
for i, idx in enumerate(top_k_indices, 1):
    top_rule = dataset[idx]
    print(f"\n{i}. Rule ID: {top_rule['rule_id']}")
    print(f"   Rule description: {top_rule['rule_description']}")
    print(f"   Similarity score: {cos_scores[idx].item():.4f}")

# should use for 1:1 mapping (hard logic tool) -> 
# 




Top 10 most relevant rules:

1. Rule ID: 22-14
   Rule description: The baseline heat-rejection device shall have a design temperature rise of 10°F.
   Similarity score: 0.7314

2. Rule ID: 22-17
   Rule description: The baseline heat rejection device shall have an efficiency of 38.2 gpm/hp
   Similarity score: 0.6872

3. Rule ID: 22-18
   Rule description: The baseline heat rejection device shall be modeled with variable speed fan control
   Similarity score: 0.6376

4. Rule ID: 22-15
   Rule description: Heat Rejection Device Approach calaculated correctly (T/F), Approach = 25.72-(0.24*WB)
   Similarity score: 0.6366

5. Rule ID: 22-13
   Rule description: The baseline heat rejection loop shall be an axial-fan open circuit cooling tower.
   Similarity score: 0.5629

6. Rule ID: 22-12
   Rule description: The heat rejection system shall be a single loop, modeled with a single cooling tower
   Similarity score: 0.5319

7. Rule ID: 22-22
   Rule description: The baseline chiller effi

#### 2.3 Use ```facebook/bart-large-cnn``` model for summarization